# Digital Biomarker Teardown — Analysis

**Thesis:** the wearable can detect the signal; reimbursement is the real constraint.

Pipeline: load -> EDA -> features -> model -> evaluation -> feature importance.

Keep the honesty note in mind: report sensitivity/specificity, not just accuracy, and be suspicious of a near-perfect AUC (it usually means leakage).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

import sys; sys.path.append('../src')
import features as F

RANDOM_STATE = 42
sns.set_theme()

## 1. Load

Download a dataset (see `data/DATA.md`), drop it in `data/raw/`, then adapt this cell to the file schema.

In [ ]:
# TODO: load raw records into a list of (subject_id, label, dataframe)
# For PhysioNet gaitndd, use wfdb or plain pd.read_csv on the .ts files.
#
# records = []  # each: {'id': str, 'label': 0/1, 'signal': pd.DataFrame}
# for path in glob('data/raw/*.ts'):
#     df = pd.read_csv(path, sep=r'\s+', header=None, names=[...])
#     records.append({'id': ..., 'label': ..., 'signal': df})
raise NotImplementedError('Wire up loading to your dataset schema')

## 2. Exploratory analysis

Honest EDA before modeling: class balance, missingness, a few distribution plots. This is the reproducible-pipeline (info-science) layer.

In [ ]:
# labels = pd.Series([r['label'] for r in records])
# print('Class balance:\n', labels.value_counts(normalize=True))
# print('N subjects:', len(records))
# TODO: plot a couple of raw stride-interval traces, healthy vs disease

## 3. Feature engineering

The intellectual core. See `src/features.py`. 8-15 domain-motivated features.

In [ ]:
# X = pd.DataFrame([F.build_feature_row(r['signal']) for r in records])
# y = labels.values
# X = X.fillna(X.median())   # decide imputation deliberately, note it in the memo
# X.describe()

## 4. Model

Two classifiers: logistic regression (interpretable baseline) and random forest (stronger). Proper split + cross-validation.

In [ ]:
# X_tr, X_te, y_tr, y_te = train_test_split(
#     X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
# scaler = StandardScaler().fit(X_tr)
# X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)

# logit = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
# rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_tr, y_tr)

# print('Logit CV AUC:', cross_val_score(logit, X_tr_s, y_tr, cv=5, scoring='roc_auc').mean())
# print('RF    CV AUC:', cross_val_score(rf, X_tr, y_tr, cv=5, scoring='roc_auc').mean())

## 5. Evaluation

Clinically meaningful metrics: ROC-AUC, sensitivity, specificity — not accuracy alone. Save the ROC curve to `reports/figures/` for the memo.

In [ ]:
# proba = rf.predict_proba(X_te)[:, 1]
# auc = roc_auc_score(y_te, proba)
# fpr, tpr, thr = roc_curve(y_te, proba)
# print(f'Test ROC-AUC: {auc:.3f}')
# print(classification_report(y_te, rf.predict(X_te)))
#
# plt.plot(fpr, tpr, label=f'RF (AUC={auc:.2f})'); plt.plot([0,1],[0,1],'--')
# plt.xlabel('1 - Specificity'); plt.ylabel('Sensitivity'); plt.legend()
# plt.savefig('../reports/figures/roc.png', dpi=150, bbox_inches='tight')

## 6. Feature importance — your 'so what'

Which gait features carried the signal? This is what you talk about in an interview.

In [ ]:
# imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
# imp.plot.barh(); plt.title('Feature importance')
# plt.savefig('../reports/figures/importance.png', dpi=150, bbox_inches='tight')
# imp.sort_values(ascending=False)

## 7. Hand-off to the economics half

The model answers *can we detect it*. The memo (`reports/memo.md`) answers *does anyone pay for it* — TAM + RPM/RTM reimbursement. That gap between clinically-detectable and reimbursable is the argument.